# Introduction to Machine Learning: Deep Learning

**Instructor:** Daniel Acuna, Ph.D.
**Position:** Associate Professor of Computer Science
**Institution:** University of Colorado Boulder

---

# Lab 4: Sequence Models and Attention

In this lab, you will explore sequential data processing using Recurrent Neural Networks (RNNs), Long Short-Term Memory networks (LSTMs), and Transformers. You will work with the IMDB dataset to perform sentiment analysis.

---

## Setup (do not edit)

In [1]:
import pathlib
import pickle
from typing import Tuple, Dict, List, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Embedding, SimpleRNN, LSTM, Dense, Bidirectional, 
    Input, Dropout, LayerNormalization, MultiHeadAttention
)

# Suppress TensorFlow warnings
tf.get_logger().setLevel("ERROR")

RANDOM_STATE: int = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

_DATA_PATH = pathlib.Path("imdb_reviews.csv")

if not _DATA_PATH.exists():
    raise FileNotFoundError(
        "imdb_reviews.csv is missing. Please run the download script (w4_download_datasets.py) "
        "or ask the TA for assistance."
    )

## 1. Load and Inspect the Dataset *(10 points)*

Load the dataset from `imdb_reviews.csv`.

1. Read the CSV into a DataFrame.
2. Display the first 5 rows (ungraded).
3. Store the shape of the DataFrame in **`q1_shape`**.
4. Store the class distribution (counts of each label) in **`q1_dist`**.

In [2]:
def load_data(path: pathlib.Path) -> pd.DataFrame:
    """Load the dataset from CSV.
    
    Parameters
    ----------
    path : pathlib.Path
        Path to the CSV file.
        
    Returns
    -------
    pd.DataFrame
        The loaded dataset.
    """
    # your code here
    #raise NotImplementedError
    return pd.read_csv(path)

df = load_data(_DATA_PATH)
q1_shape: Tuple[int, int] = df.shape
q1_dist: pd.Series = df['label'].value_counts().sort_index()

print(f"Shape: {q1_shape}")
print(f"Class distribution:\n{q1_dist}")
df.head()

Shape: (10000, 2)
Class distribution:
label
0    4912
1    5088
Name: count, dtype: int64


,text,label
0,? never mind the serious logic gaps never mind...,0
1,? overall this movie is dreadful and should ha...,0
2,? this movie is about ? a gladiator who is bro...,0
3,? drawing restraint 9 ? matthew barney br br h...,0
4,? this story had a good plot to it about four ...,0


In [3]:
# If all tests pass (there might be hidden tests), you will earn 10 points
assert isinstance(df, pd.DataFrame), "df should be a DataFrame"
assert isinstance(q1_shape, tuple), "q1_shape must be a tuple"
assert len(q1_shape) == 2, "q1_shape must have 2 dimensions"
assert "text" in df.columns and "label" in df.columns, "DataFrame must have 'text' and 'label' columns"
assert isinstance(q1_dist, pd.Series), "q1_dist must be a pandas Series"


## 2. Tokenization and Padding *(10 points)*

Neural networks cannot work directly with text strings. We need to:
1. Tokenize: Convert words to integers.
2. Pad: Ensure all sequences have the same length.

**Task:**
1. Initialize a `Tokenizer` with `num_words=10000` and `oov_token='<OOV>'`.
2. Fit the tokenizer on `df['text']`.
3. Convert texts to sequences of integers.
4. Pad the sequences to `maxlen=100` using `padding='post'` and `truncating='post'`.
5. Split the data into training (80%) and test (20%) sets using `train_test_split` with `random_state=RANDOM_STATE`.

Store:
- **`X_train`**, **`X_test`**: Padded sequences (numpy arrays)
- **`y_train`**, **`y_test`**: Labels (numpy arrays)
- **`vocab_size`**: The actual vocabulary size (should be 10000)

*Note:* We use a smaller vocabulary and sequence length to keep training fast.

In [4]:
# Global parameters
VOCAB_SIZE = 10000
MAX_LEN = 100
EMBEDDING_DIM = 32

# your code here
#raise NotImplementedError
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df['text'])
sequences = tokenizer.texts_to_sequences(df['text'])
padded_sequences = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
labels = df['label'].values
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Testing set shape: {X_test.shape}, {y_test.shape}")

Training set shape: (8000, 100), (8000,)
Testing set shape: (2000, 100), (2000,)


In [5]:
assert isinstance(X_train, np.ndarray), "X_train must be a numpy array"
assert X_train.shape[1] == MAX_LEN, f"Sequence length must be {MAX_LEN}"
assert X_train.shape[0] + X_test.shape[0] == len(df), "Train + Test size should equal total samples"
assert len(tokenizer.word_index) > 0, "Tokenizer should be fitted"


## 3. Build a Simple RNN Model *(10 points)*

Recurrent Neural Networks (RNNs) process sequences by maintaining a hidden state.

Build a model with:
1. `Input` layer: shape=(MAX_LEN,)
2. `Embedding` layer: input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM
3. `SimpleRNN` layer: 32 units
4. `Dense` layer: 1 unit, sigmoid activation (for binary classification)

Use `model.add()` to add each layer. Start with `Input(shape=(MAX_LEN,))` to define the input shape.

Compile with `adam` optimizer, `binary_crossentropy` loss, and `accuracy` metric.

Store the model in **`model_rnn`**.

In [9]:
# your code here
model_rnn = Sequential()
model_rnn.add(Input(shape=(MAX_LEN,)))
model_rnn.add(Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM))
model_rnn.add(SimpleRNN(32))
model_rnn.add(Dense(1, activation='sigmoid'))

model_rnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
 )

model_rnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 322,113 (1.23 MB)

 Trainable params: 322,113 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
assert isinstance(model_rnn, Sequential), "model_rnn must be a Sequential model"
assert len(model_rnn.layers) >= 3, "Model should have at least 3 layers"
assert isinstance(model_rnn.layers[0], Embedding), "First layer must be Embedding"
assert isinstance(model_rnn.layers[1], SimpleRNN), "Second layer must be SimpleRNN"
assert model_rnn.output_shape == (None, 1), "Output shape should be (None, 1)"


## 4. Train the Simple RNN *(10 points)*

Train `model_rnn` for **5 epochs** with `batch_size=64` using `X_train` and `y_train`.
Use `X_test` and `y_test` for validation.

Store the history in **`history_rnn`** and the final validation accuracy (rounded to 3 decimals) in **`q4_val_acc`**.

In [13]:
# your code here
history_rnn = model_rnn.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1
)

q4_val_acc = round(history_rnn.history['val_accuracy'][-1], 3)

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.9277 - loss: 0.2044 - val_accuracy: 0.7735 - val_loss: 0.5555
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9797 - loss: 0.0811 - val_accuracy: 0.7610 - val_loss: 0.6147
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9778 - loss: 0.0744 - val_accuracy: 0.7320 - val_loss: 0.6941
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9904 - loss: 0.0422 - val_accuracy: 0.7595 - val_loss: 0.7448
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9966 - loss: 0.0202 - val_accuracy: 0.7380 - val_loss: 0.8905


In [15]:
assert 'val_accuracy' in history_rnn.history, "History should contain val_accuracy"
assert len(history_rnn.history['loss']) == 5, "Should train for 5 epochs"
print(f"SimpleRNN Val Acc: {q4_val_acc}")


SimpleRNN Val Acc: 0.738


## 5. Build an LSTM Model *(10 points)*

LSTMs are better at capturing long-term dependencies than SimpleRNNs.

Build a model with:
1. `Input` layer: shape=(MAX_LEN,)
2. `Embedding` layer: input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM
3. `LSTM` layer: 32 units
4. `Dense` layer: 1 unit, sigmoid activation

Use `model.add()` to add each layer.

Compile with `adam`, `binary_crossentropy`, and `accuracy`.

Store the model in **`model_lstm`**.

In [16]:
# your code here
model_lstm = Sequential()
model_lstm.add(Input(shape=(MAX_LEN,)))
model_lstm.add(Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM))
model_lstm.add(LSTM(32))
model_lstm.add(Dense(1, activation='sigmoid'))

model_lstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
 )

model_lstm.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 100, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 328,353 (1.25 MB)

 Trainable params: 328,353 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
assert isinstance(model_lstm, Sequential), "model_lstm must be a Sequential model"
assert isinstance(model_lstm.layers[1], LSTM), "Second layer must be LSTM"


## 6. Train the LSTM *(10 points)*

Train `model_lstm` for **5 epochs** with `batch_size=64`.
Use validation data.

Store history in **`history_lstm`** and final validation accuracy in **`q6_val_acc`**.

In [18]:
# your code here
history_lstm = model_lstm.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1
)

q6_val_acc = round(history_lstm.history['val_accuracy'][-1], 3)

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.5914 - loss: 0.6640 - val_accuracy: 0.7380 - val_loss: 0.5459
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.8101 - loss: 0.4297 - val_accuracy: 0.7975 - val_loss: 0.4713
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.8930 - loss: 0.2898 - val_accuracy: 0.7980 - val_loss: 0.5270
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.8921 - loss: 0.3102 - val_accuracy: 0.7690 - val_loss: 0.5375
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9276 - loss: 0.2310 - val_accuracy: 0.7850 - val_loss: 0.5911


In [19]:
assert 'val_accuracy' in history_lstm.history, "History should contain val_accuracy"
print(f"LSTM Val Acc: {q6_val_acc}")


LSTM Val Acc: 0.785


## 7. Bidirectional LSTM *(10 points)*

Bidirectional RNNs process the sequence in both directions (forward and backward), often improving performance.

Build a model with:
1. `Input` layer: shape=(MAX_LEN,)
2. `Embedding` layer
3. `Bidirectional` wrapper around an `LSTM(32)` layer
4. `Dense` layer (sigmoid)

Use `model.add()` to add each layer.

Train for **5 epochs**.

Store model in **`model_bilstm`**, history in **`history_bilstm`**, and final validation accuracy in **`q7_val_acc`**.

In [20]:
# your code here
model_bilstm = Sequential()
model_bilstm.add(Input(shape=(MAX_LEN,)))
model_bilstm.add(Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM))
model_bilstm.add(Bidirectional(LSTM(32)))
model_bilstm.add(Dense(1, activation='sigmoid'))

model_bilstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
 )

history_bilstm = model_bilstm.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1
)

q7_val_acc = round(history_bilstm.history['val_accuracy'][-1], 3)

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.6535 - loss: 0.6065 - val_accuracy: 0.7960 - val_loss: 0.4492
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.8499 - loss: 0.3583 - val_accuracy: 0.8045 - val_loss: 0.4779
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9039 - loss: 0.2624 - val_accuracy: 0.7975 - val_loss: 0.4353
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9274 - loss: 0.2148 - val_accuracy: 0.7945 - val_loss: 0.5172
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9358 - loss: 0.1777 - val_accuracy: 0.8100 - val_loss: 0.5258


In [21]:
assert isinstance(model_bilstm.layers[1], Bidirectional), "Second layer must be Bidirectional"
# Check that it wraps an LSTM (works with both Keras 2 and 3)
wrapped_layer = getattr(model_bilstm.layers[1], 'forward_layer', None) or getattr(model_bilstm.layers[1], 'layer', None)
assert wrapped_layer is not None and isinstance(wrapped_layer, LSTM), "Wrapped layer must be LSTM"
print(f"Bidirectional LSTM Val Acc: {q7_val_acc}")


Bidirectional LSTM Val Acc: 0.81


## 8. Transformer Block *(10 points)*

Transformers use self-attention to process sequences in parallel.

Create a custom `TransformerBlock` layer (or use Keras functional API) to build a small Transformer model.

Architecture:
1. Input layer (shape=`(MAX_LEN,)`)
2. Embedding layer
   - Use `Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN)`
3. MultiHeadAttention layer
   - Use `MultiHeadAttention(num_heads=2, key_dim=EMBEDDING_DIM)`
   - Note: In self-attention, the query, key, and value are the same tensor (the output of embedding).
     Pass `(embedding_output, embedding_output)` as arguments.
4. GlobalAveragePooling1D
   - To flatten the sequence into a single vector
5. Dropout (0.1)
6. Dense (20, relu)
7. Dropout (0.1)
8. Dense (1, sigmoid)

*Note: A full transformer usually includes LayerNormalization and skip connections, but we'll stick to a simplified version for this exercise.*

Implement this using the Functional API.

Store the model in **`model_transformer`**.

In [22]:
from tensorflow.keras.layers import GlobalAveragePooling1D

# your code here
transformer_inputs = Input(shape=(MAX_LEN,))
x = Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN)(transformer_inputs)
x = MultiHeadAttention(num_heads=2, key_dim=EMBEDDING_DIM)(x, x)
x = GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)
x = Dense(20, activation='relu')(x)
x = Dropout(0.1)(x)
transformer_outputs = Dense(1, activation='sigmoid')(x)

model_transformer = Model(inputs=transformer_inputs, outputs=transformer_outputs)
model_transformer.summary()

c:\Users\garey\anaconda3\envs\tf220\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 100, 32)   │    320,000 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 100, 32)   │      8,416 │ embedding_4[0][0… │
│ (MultiHeadAttentio… │                   │            │ embedding_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ multi_head_atten… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 32)        │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 20)        │        660 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 20)        │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         21 │ dropout_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 329,097 (1.26 MB)

 Trainable params: 329,097 (1.26 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
assert isinstance(model_transformer, Model), "model_transformer must be a Keras Model"
# Check for MultiHeadAttention layer
assert any(isinstance(l, MultiHeadAttention) for l in model_transformer.layers), "Must include MultiHeadAttention layer"


## 9. Train Transformer *(10 points)*

Train the transformer model for **5 epochs** with `batch_size=64`.

Store history in **`history_transformer`** and final validation accuracy in **`q9_val_acc`**.

In [24]:
# your code here
model_transformer.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
 )

history_transformer = model_transformer.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1
)

q9_val_acc = round(history_transformer.history['val_accuracy'][-1], 3)

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.6499 - loss: 0.6181 - val_accuracy: 0.7855 - val_loss: 0.4462
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8500 - loss: 0.3488 - val_accuracy: 0.8120 - val_loss: 0.4240
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.9120 - loss: 0.2355 - val_accuracy: 0.8070 - val_loss: 0.5458
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9475 - loss: 0.1645 - val_accuracy: 0.7865 - val_loss: 0.7541
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.9588 - loss: 0.1317 - val_accuracy: 0.7915 - val_loss: 0.7961


In [25]:
print(f"Transformer Val Acc: {q9_val_acc}")
assert 'val_accuracy' in history_transformer.history


Transformer Val Acc: 0.791


## 10. Model Comparison *(10 points)*

Compare the validation accuracies of all four models.

Create a dictionary **`q10_results`** with keys: `'SimpleRNN'`, `'LSTM'`, `'BiLSTM'`, `'Transformer'` and their corresponding final validation accuracies.

Which model performed best? Store the name of the best model in **`q10_best_model`**.

In [26]:
# your code here
q10_results = {
    'SimpleRNN': q4_val_acc,
    'LSTM': q6_val_acc,
    'BiLSTM': q7_val_acc,
    'Transformer': q9_val_acc
}

q10_best_model = max(q10_results, key=q10_results.get)

In [27]:
assert isinstance(q10_results, dict), "q10_results must be a dictionary"
assert len(q10_results) == 4, "Must compare 4 models"
assert isinstance(q10_best_model, str), "q10_best_model must be a string"

print("Model Comparison:")
for model, acc in q10_results.items():
    print(f"{model}: {acc}")
print(f"Best Model: {q10_best_model}")

Model Comparison:
SimpleRNN: 0.738
LSTM: 0.785
BiLSTM: 0.81
Transformer: 0.791
Best Model: BiLSTM


## Next Steps

Congratulations on completing the assignment! Before submitting:

1. Make sure all your cells run without errors.
2. Ensure you've answered all parts of each question.
3. If any autograder tests fail, revisit your answers.
